# TD07 — Évaluer un modèle, courbes d'apprentissage et élan

## Objectif

Ce TD prolonge le TD03 (MLP et rétropropagation) et le chapitre « 6 - Évaluer un modèle ». On y **mesure** la qualité d'un modèle au lieu de la supposer : séparation entraînement / validation / test, courbes d'apprentissage, choix du nombre d'époques, et métriques adaptées aux classes déséquilibrées.

On étudie ensuite l'**élan** (momentum) : son effet sur la vitesse d'apprentissage et sur le surapprentissage, lu sur les courbes d'apprentissage.

Les données sont générées par des fonctions fournies (blobs déséquilibrés, lunes, spirales) ; le MLP fourni est un perceptron multicouche à activations tanh (couches cachées) et sigmoïde (sortie).

## Préambule

Les cellules ci-dessous sont fournies : alias de types, générateurs de données, classe `MLP` (même structure qu'au TD03, activations figées) et fonctions de tracé.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Alias de types pour les tableaux numpy
Vec = np.ndarray[tuple[int], np.dtype[np.float64]]        # vecteur 1D de flottants
Mat = np.ndarray[tuple[int, int], np.dtype[np.float64]]   # matrice 2D de flottants

### Générateurs de données

Chaque générateur renvoie `X` de forme $(n, 2)$ et les cibles `t` de forme $(n, 1)$, en flottants `0.` / `1.`. L'argument `seed` fixe le tirage.

In [ ]:
def make_blobs_imbalanced(n: int, noise: float = 0.6, pos_frac: float = 0.1, seed: int = 1) -> tuple[Mat, Mat]:
    """Deux nuages gaussiens, classe 1 minoritaire.

    Classe 0 centrée en (0, 0), classe 1 centrée en (1, 1) ; chaque point est
    de classe 1 avec la probabilité ``pos_frac``.

    Returns
    -------
    tuple[Mat, Mat]
        X de forme (n, 2) et t de forme (n, 1).
    """
    rng = np.random.default_rng(seed)
    lab = (rng.uniform(size=n) < pos_frac).astype(float)
    centers = np.where(lab[:, None] == 1, [1.0, 1.0], [0.0, 0.0])
    X = centers + rng.normal(0.0, noise, size=(n, 2))
    return X, lab[:, None]


def make_moons(n: int, noise: float = 0.3, seed: int = 1) -> tuple[Mat, Mat]:
    """Deux demi-lunes entrelacées, bruit gaussien d'écart-type ``noise``.

    Returns
    -------
    tuple[Mat, Mat]
        X de forme (n, 2) et t de forme (n, 1).
    """
    rng = np.random.default_rng(seed)
    a = rng.uniform(0.0, np.pi, size=n)
    lab = rng.integers(0, 2, size=n)
    x = np.where(lab == 0, np.cos(a), 1.0 - np.cos(a))
    y = np.where(lab == 0, np.sin(a), 0.5 - np.sin(a))
    X = np.c_[x, y] + rng.normal(0.0, noise, size=(n, 2))
    return X, lab[:, None].astype(float)


def make_spirals(n: int, noise: float = 0.08, seed: int = 1) -> tuple[Mat, Mat]:
    """Deux spirales entrelacées (rayon r dans [0.2, 1], angle 3πr, +π pour la classe 1).

    Returns
    -------
    tuple[Mat, Mat]
        X de forme (n, 2) et t de forme (n, 1).
    """
    rng = np.random.default_rng(seed)
    lab = rng.integers(0, 2, size=n)
    r = rng.uniform(0.2, 1.0, size=n)
    theta = 3 * np.pi * r + lab * np.pi
    X = np.c_[r * np.cos(theta), r * np.sin(theta)] + rng.normal(0.0, noise, size=(n, 2))
    return X, lab[:, None].astype(float)

### Le réseau

`MLP(sizes, seed)` : couches cachées en tanh, sortie sigmoïde, poids initialisés selon $\mathcal N(0, 0.5)$, biais nuls. Le coût est celui du cours, $E = \tfrac12\,\operatorname{mean}\sum_k (Y_k - T_k)^2$, et `error_rate` donne la proportion d'exemples mal classés au seuil choisi.

`fit` effectue une étape de descente de gradient par époque (sur tout le jeu) et enregistre l'historique **après** chaque étape : l'entrée `i` de l'historique correspond à l'état du modèle après `i + 1` étapes. Les clés `"val"` et `"err_val"` sont des listes vides si aucun jeu de validation n'est fourni.

In [ ]:
class MLP:
    def __init__(self, sizes: list[int], seed: int = 0):
        rng = np.random.default_rng(seed)
        self.sizes = list(sizes)
        self.Ws: list[Mat] = []
        self.bs: list[Vec] = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.Ws.append(rng.normal(0.0, 0.5, size=(n_out, n_in)))
            self.bs.append(np.zeros(n_out))

    def forward(self, X: Mat) -> tuple[list[Mat], list[Mat]]:
        """Passe avant : renvoie (As, Zs), activations (avec Y0 = X en tête) et pré-activations."""
        A = X
        As = [A]
        Zs = []
        n = len(self.Ws)
        for l, (W, b) in enumerate(zip(self.Ws, self.bs)):
            Z = A @ W.T + b
            Zs.append(Z)
            A = 1.0 / (1.0 + np.exp(-Z)) if l == n - 1 else np.tanh(Z)
            As.append(A)
        return As, Zs

    def backward(self, X: Mat, t: Mat) -> list[tuple[Mat, Vec]]:
        """Passe arrière : liste de (dW_l, db_l), une paire par couche."""
        As, Zs = self.forward(X)
        N = X.shape[0]
        Y = As[-1]
        delta = (Y - t) * Y * (1.0 - Y)
        grads: list[tuple[Mat, Vec]] = []
        for l in reversed(range(len(self.Ws))):
            dW = delta.T @ As[l] / N
            db = delta.mean(axis=0)
            grads.append((dW, db))
            if l > 0:
                delta = (delta @ self.Ws[l]) * (1.0 - np.tanh(Zs[l - 1]) ** 2)
        grads.reverse()
        return grads

    def step(self, X: Mat, t: Mat, lr: float) -> None:
        """Une étape de descente de gradient sur le jeu (X, t)."""
        for l, (dW, db) in enumerate(self.backward(X, t)):
            self.Ws[l] = self.Ws[l] - lr * dW
            self.bs[l] = self.bs[l] - lr * db

    def predict(self, X: Mat) -> Mat:
        """Probabilités de la classe 1, de forme (N, 1)."""
        return self.forward(X)[0][-1]

    def loss(self, X: Mat, t: Mat) -> float:
        """Coût : E = (1/2) · mean(Σ_k (Y_k - T_k)²)."""
        return 0.5 * float(np.mean(np.sum((self.predict(X) - t) ** 2, axis=1)))

    def error_rate(self, X: Mat, t: Mat, threshold: float = 0.5) -> float:
        """Proportion d'exemples mal classés (classe 1 prédite si la probabilité dépasse ``threshold``)."""
        return float(np.mean((self.predict(X) > threshold) != (t > 0.5)))

    def fit(self, X: Mat, t: Mat, lr: float, epochs: int,
            X_val: Mat | None = None, t_val: Mat | None = None) -> dict[str, list[float]]:
        """Entraîne le réseau (une étape par époque) et renvoie l'historique.

        Clés : ``"train"``, ``"val"`` (coûts) et ``"err_train"``, ``"err_val"``
        (taux d'erreur). L'entrée i est mesurée après i + 1 étapes ; les listes
        ``"val"`` et ``"err_val"`` restent vides sans jeu de validation.
        """
        history: dict[str, list[float]] = {"train": [], "val": [], "err_train": [], "err_val": []}
        has_val = X_val is not None and t_val is not None
        for _ in range(epochs):
            self.step(X, t, lr)
            history["train"].append(self.loss(X, t))
            history["err_train"].append(self.error_rate(X, t))
            if has_val:
                history["val"].append(self.loss(X_val, t_val))
                history["err_val"].append(self.error_rate(X_val, t_val))
        return history

### Tracés

`plot_curves` superpose les courbes d'apprentissage de plusieurs entraînements (une couleur par entrée ; train en pointillés, validation en trait plein). Avec `metric="err"`, on trace les taux d'erreur au lieu du coût. `plot_data` affiche un nuage de points coloré par classe, `plot_decision` ajoute la frontière de décision d'un modèle (probabilité 0.5).

In [ ]:
def plot_curves(histories: dict[str, dict[str, list[float]]], metric: str = "loss",
                logx: bool = True, ax=None):
    """Courbes d'apprentissage de plusieurs historiques sur un même graphique.

    ``metric="loss"`` trace le coût, ``metric="err"`` les taux d'erreur.
    """
    if ax is None:
        fig, ax = plt.subplots()
    k_tr, k_va = ("train", "val") if metric == "loss" else ("err_train", "err_val")
    for i, (name, h) in enumerate(histories.items()):
        c = f"C{i}"
        x = np.arange(1, len(h[k_tr]) + 1)
        ax.plot(x, h[k_tr], color=c, ls="--", lw=1)
        if h[k_va]:
            ax.plot(x, h[k_va], color=c, label=name)
        else:
            ax.plot([], [], color=c, ls="--", label=name)
    ax.set_xlabel("époque")
    ax.set_ylabel("coût" if metric == "loss" else "taux d'erreur")
    if logx:
        ax.set_xscale("log")
    if metric == "loss":
        ax.set_yscale("log")
    ax.legend()
    ax.grid(True)
    return ax


def plot_data(X: Mat, t: Mat, ax=None):
    """Nuage de points coloré par classe."""
    if ax is None:
        fig, ax = plt.subplots()
    ax.scatter(X[:, 0], X[:, 1], c=t.ravel(), cmap="RdBu", vmin=0, vmax=1,
               edgecolors="k", s=25, zorder=3)
    ax.set_aspect("equal")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.grid(True)
    return ax


def plot_decision(model: MLP, X: Mat, t: Mat, ax=None):
    """Probabilité prédite (fond), frontière à 0.5 (trait noir) et points (X, t)."""
    if ax is None:
        fig, ax = plt.subplots()
    m = 0.3
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - m, X[:, 0].max() + m, 200),
                         np.linspace(X[:, 1].min() - m, X[:, 1].max() + m, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 21), cmap="RdBu", alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=2)
    plot_data(X, t, ax=ax)
    return ax

### Vérification du réseau

La cellule suivante compare le gradient de `backward` à un gradient numérique (différences finies) sur un petit réseau, et vérifie la longueur de l'historique renvoyé par `fit`.

In [ ]:
def check_mlp(atol: float = 1e-6, eps: float = 1e-6):
    model = MLP([2, 3, 1], seed=1)
    X = np.random.default_rng(0).normal(size=(5, 2))
    t = (np.random.default_rng(1).uniform(size=(5, 1)) > 0.5).astype(float)

    grads = model.backward(X, t)
    for l in range(len(model.Ws)):
        for params, g in ((model.Ws[l], grads[l][0]), (model.bs[l], grads[l][1])):
            num = np.zeros_like(params)
            for idx in np.ndindex(*params.shape):
                old = params[idx]
                params[idx] = old + eps
                e_plus = model.loss(X, t)
                params[idx] = old - eps
                e_moins = model.loss(X, t)
                params[idx] = old
                num[idx] = (e_plus - e_moins) / (2 * eps)
            assert np.allclose(g, num, atol=atol), f"gradient incorrect (couche {l})"

    h = MLP([2, 3, 1], seed=1).fit(X, t, lr=0.1, epochs=3, X_val=X, t_val=t)
    for k in ("train", "val", "err_train", "err_val"):
        assert len(h[k]) == 3, f"historique '{k}' de longueur {len(h[k])} au lieu de 3"
    h = MLP([2, 3, 1], seed=1).fit(X, t, lr=0.1, epochs=3)
    assert len(h["train"]) == 3 and h["val"] == [] and h["err_val"] == [], "historique sans validation incorrect"
    print("MLP check — OK")


check_mlp()